# Lab 1 — From chatbot to agent (EP03)

In the theory video we saw that an **agent** is an LLM working in a loop — *perceive → reason → act* — through **tools**,
with a **harness** around it. In this lab we build an AI301 course assistant that can answer
*"how many days until the midterm?"* and **actually book a tutoring room** — all with our own code.

**What you'll learn**
1. chatbot: one model call — it can't answer and can't book
2. tools: calendar, free-room and booking functions + the JSON schemas we give the model
3. the agent loop: our first harness (loop until done, or until it has to ask a person)
4. persistence: keep the history in a list, so the conversation can continue across turns
5. safety: a step limit + a person approves before booking (human-in-the-loop)

The calendar and rooms in this lab are **simulated data written for teaching** (today = 10 Nov 2026);
every model answer and tool call comes from a real run.
(The video uses Thai prompts; this English notebook uses English ones, so your outputs will differ in wording.)

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, ch15 — CC BY-SA 4.0

## 0. Set up before you start

| What you need | Details |
|---|---|
| Python 3.10 or later | or use Google Colab, which already has it |
| OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs well under one cent) |
| Packages | `openai`, `python-dotenv` (installed in the next cell) |

**Add your API key**
- **Colab:** 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file in the same folder as the notebook and add the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key directly in your code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. Calling Gemma through OpenRouter

**Why `google/gemma-4-26b-a4b-it`** — follows instructions well and **can call tools** (tool calling),
which is the heart of the agent in this lab and in Lab 2

**Why we pin the provider to DeepInfra** — Parasail, used in EP02, doesn't support tools at all.
We tested this lab's agent on 10 providers that support tools (2026-11-10): most took the same steps every time,
but one read "this Thursday" as a different day → we pin one provider (`deepinfra/fp8`) so results repeat

**Not free**, but very cheap: one full run of the lab ≈ 5,800 tokens ≈ $0.0006
· the `:free` version doesn't work (tested 2026-11-10: 4 of 4 calls got 429 "rate-limited upstream")

In [ ]:
import json, os
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["deepinfra/fp8"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 2. As a chatbot: one model call

Send the student's request straight to the model — no tools, no loop

In [ ]:
QUESTION = "How many days until the AI301 midterm? And please book me a tutoring room this Thursday afternoon."
reply = client.chat.completions.create(model=MODEL, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": QUESTION}])
print(reply.choices[0].message.content)

## 3. Tools: the calendar and the booking system

Simulated data first: the course calendar (including today's date) and the tutoring rooms that are still free

In [ ]:
CALENDAR = {"today": "2026-11-10 (Tuesday)",
            "events": [{"name": "AI301 midterm exam", "date": "2026-11-18 (Wednesday)"},
                       {"name": "Homework 3 due", "date": "2026-11-14 (Saturday)"}]}
FREE = {"2026-11-12": {"Room 1": ["09:00"], "Room 3": ["13:00", "15:00"], "Room 5": []},
        "2026-11-13": {"Room 1": ["10:00", "13:00"], "Room 3": [], "Room 5": ["15:00"]}}
BOOKING = {}

In [ ]:
def get_calendar():
    return CALENDAR

def find_rooms(date):
    if date not in FREE:
        return {"error": f"No room data for {date} (available: {', '.join(FREE)})"}
    return {"date": date, "free": {r: t for r, t in FREE[date].items() if t}}

def book_room(room, date, time):
    if time not in FREE.get(date, {}).get(room, []):
        return {"ok": False, "error": f"{room} on {date} at {time} is not free"}
    BOOKING.clear(); BOOKING.update(room=room, date=date, time=time)
    return {"ok": True, "booking": dict(BOOKING)}

**Tell the model which tools exist** — each tool is a JSON schema: name · what it does · which arguments it takes

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_calendar", "description": "Get today's date and the AI301 exam and due-date calendar",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "find_rooms", "description": "List tutoring rooms that are free on a given date",
        "parameters": {"type": "object", "required": ["date"],
                       "properties": {"date": {"type": "string", "description": "YYYY-MM-DD"}}}}},
    {"type": "function", "function": {
        "name": "book_room", "description": "Book a tutoring room (one booking at a time; a new booking replaces the old one)",
        "parameters": {"type": "object", "required": ["room", "date", "time"],
                       "properties": {"room": {"type": "string"}, "date": {"type": "string", "description": "YYYY-MM-DD"},
                                      "time": {"type": "string", "description": "HH:MM"}}}}},
]

In [ ]:
FUNCS = {"get_calendar": get_calendar, "find_rooms": find_rooms, "book_room": book_room}

def run_tool(call):
    args = json.loads(call.function.arguments or "{}")
    return json.dumps(FUNCS[call.function.name](**args), ensure_ascii=False)

## 4. The agent loop = our first harness

| In the loop | From the theory video |
|---|---|
| send `history` + `TOOLS` to the model | 💭 reason |
| the model asks for a tool → `run_tool` | 🔧 act |
| the tool's result goes back into `history` | 👀 perceive (observation) |
| no more tool requests → answer / ask a person | 🏁 done / 🙋 ask a person |
| `max_steps` | 🛑 guardrail: how many rounds at most |

In [ ]:
SYSTEM = "You are the AI301 course assistant. Help students check the calendar and book tutoring rooms. Use tools to get real data; never guess dates."
MAX_STEPS = 8
NEEDS_APPROVAL = set()   # part 6 adds the tools that need a person's approval
PENDING = []

In [ ]:
def call_llm(history):
    resp = client.chat.completions.create(model=MODEL, messages=history, tools=TOOLS, extra_body=EXTRA)
    return resp.choices[0].message

In [ ]:
def run_agent(history, max_steps=MAX_STEPS):
    for step in range(1, max_steps + 1):
        msg = call_llm(history)
        if not msg.tool_calls:
            history.append({"role": "assistant", "content": msg.content})
            return msg.content
        history.append({"role": "assistant", "content": msg.content or "",
                        "tool_calls": [c.model_dump() for c in msg.tool_calls]})
        for call in msg.tool_calls:
            print(f"  🔧 {call.function.name}({call.function.arguments})")
            if call.function.name in NEEDS_APPROVAL:
                PENDING.append(call)
                continue
            result = run_tool(call)
            print(f"  👀 {result}")
            history.append({"role": "tool", "tool_call_id": call.id, "content": result})
        if PENDING:
            return "⏸ waiting for approval"
    return f"⚠️ stopped: reached {max_steps} steps"

In [ ]:
history = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}]
print(run_agent(history))

## 5. Persistence: the history is the agent's memory

The model remembers nothing between calls — all it knows is what's in `history`

In [ ]:
history.append({"role": "user", "content": "13:00, please"})
print(run_agent(history))
print(BOOKING)

In [ ]:
fresh = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Can we move it to 3 pm?"}]
print(run_agent(fresh))

## 6. Safety: guardrails and human-in-the-loop

In part 5 the agent **booked at once** — correctly this time, but if the model misreads us, a wrong booking is a real consequence.
The book says autonomous action needs guardrails and **human oversight** → the harness enforces it, instead of hoping the model asks

| Guardrail in this lab | Where |
|---|---|
| at most `MAX_STEPS` rounds | the loop |
| book only slots that are really free | `book_room` |
| `book_room` needs a person's approval first | `NEEDS_APPROVAL` + `approve()` |

In [ ]:
print(run_agent([{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}], max_steps=1))

In [ ]:
NEEDS_APPROVAL = {"book_room"}

def approve(history, ok=True):
    while PENDING:
        call = PENDING.pop(0)
        result = run_tool(call) if ok else json.dumps({"ok": False, "error": "the user did not approve"}, ensure_ascii=False)
        print(f"  ✅ {call.function.name} → {result}")
        history.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return run_agent(history)

In [ ]:
history.append({"role": "user", "content": "Can we move it to 3 pm?"})
print(run_agent(history))
print(BOOKING)

In [ ]:
print(approve(history, ok=True))
print(BOOKING)

## Summary: the 5 challenges in our code

| Challenge (ch15) | In this lab |
|---|---|
| 🧠 Persistence | the `history` list — the conversation continues · fresh history = no memory |
| 📚 Grounding | `get_calendar` gives today's date and the exam date → "8 days" |
| 🔧 Action | `find_rooms`, `book_room` + tool schemas + `run_tool` |
| 👥 Coordination | — (a single agent · EP15–16) |
| 🛡️ Safety | `MAX_STEPS` · `book_room` checks before booking · `NEEDS_APPROVAL` + `approve()` |

**Try next:** add a `cancel_booking` tool and require approval for it too ·
try `approve(history, ok=False)` and see how the agent replies

**Next video — Lab 2:** we build the same assistant with LangChain (`@tool`, `create_agent`, a checkpointer, human-in-the-loop)
and compare it with our own code piece by piece